# Module 10: Preparing Data for First-Level GLM Analysis

This notebook walks through the complete first-level GLM pipeline for a task-based fMRI emotion regulation experiment. We will load preprocessed BOLD data, build a design matrix with HRF-convolved condition regressors, select confounds, fit a `FirstLevelModel` using nilearn, and compute contrast maps for the key contrasts of interest.

**Task:** Emotion regulation — participants view negative and neutral images and either passively look at them, cognitively reappraise them, or suppress their emotional response.

### Learning Objectives

1. Understand the structure of a GLM design matrix (`Y = Xβ + ε`)
2. Build a design matrix with `nilearn.glm.first_level.make_first_level_design_matrix()`
3. Apply HRF convolution to condition regressors
4. Select and clean confound regressors from fMRIPrep output
5. Fit a `FirstLevelModel` to preprocessed BOLD data
6. Specify contrasts as linear combinations of design matrix columns
7. Compute z-maps from fitted GLM contrasts
8. Quality-check the design matrix (rank, VIF, condition coverage)

## 1  The First-Level GLM

### The General Linear Model

The first-level GLM models the BOLD timeseries at every voxel independently:

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

- $\mathbf{y} \in \mathbb{R}^T$ — BOLD signal at one voxel across $T$ timepoints
- $\mathbf{X} \in \mathbb{R}^{T \times p}$ — **design matrix** with $p$ regressors
- $\boldsymbol{\beta} \in \mathbb{R}^p$ — **parameter estimates** (effect sizes per regressor)
- $\boldsymbol{\varepsilon}$ — residuals, assumed $\sim \mathcal{N}(0, \sigma^2 \mathbf{V})$

The ordinary least-squares solution is $\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$.

### The Haemodynamic Response Function (HRF)

Neural activity triggers a delayed vascular response captured by the HRF. Each condition regressor is constructed by **convolving** a boxcar function (1 during stimulus, 0 otherwise) with the HRF:

$$x_{\text{cond}}(t) = \text{boxcar}(t) * \text{hrf}(t)$$

The SPM canonical HRF peaks at ~5 s and returns to baseline at ~20 s after stimulus onset.

### Design Matrix Components

| Component | Description |
|---|---|
| Condition regressors | HRF-convolved boxcar for each trial type |
| Drift regressors | Cosine basis functions for low-frequency noise |
| Motion parameters | 6 rigid-body parameters (trans/rot) from fMRIPrep |
| Tissue signals | Global signal, white matter, CSF |
| aCompCor | Anatomical CompCor components |

### Contrasts

A contrast $\mathbf{c}$ is a vector of weights over the design matrix columns. The contrast t-statistic is:

$$t = \frac{\mathbf{c}'\hat{\boldsymbol{\beta}}}{\sqrt{\mathbf{c}'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{c} \cdot \hat{\sigma}^2}}$$

For this task, our contrasts of interest are:
- **Reappraise vs Look_Neg** — regions engaged by cognitive reappraisal
- **Suppress vs Look_Neg** — regions engaged by expressive suppression
- **Reappraise vs Suppress** — differential engagement of regulation strategies

## 2  Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuration — edit these paths to point to your data.
# ---------------------------------------------------------------------------
SUBJECT       = "sub-01"
TASK          = "emotionreg"
SPACE         = "MNI152NLin2009cAsym"
TR            = 2.0           # repetition time in seconds
FWHM          = 0.0           # spatial smoothing (mm); 0 = no additional smoothing
HRF_MODEL     = "spm"         # HRF model: 'spm', 'spm + derivative', 'glover'
HIGH_PASS     = 128.0         # high-pass filter period in seconds
FD_THRESHOLD  = 0.5           # framewise displacement threshold (mm) for scrubbing

BIDS_DIR      = os.path.abspath("../data/bids")
FMRIPREP_DIR  = os.path.abspath("../data/derivatives/fmriprep")
OUTPUT_DIR    = os.path.abspath("../data/derivatives/glm_first_level")

# Resolve expected file paths
BOLD_PATH = os.path.join(
    FMRIPREP_DIR, SUBJECT, "func",
    f"{SUBJECT}_task-{TASK}_space-{SPACE}_desc-preproc_bold.nii.gz",
)
EVENTS_PATH = os.path.join(
    BIDS_DIR, SUBJECT, "func",
    f"{SUBJECT}_task-{TASK}_events.tsv",
)
CONFOUNDS_PATH = os.path.join(
    FMRIPREP_DIR, SUBJECT, "func",
    f"{SUBJECT}_task-{TASK}_desc-confounds_timeseries.tsv",
)

# Add repo root to sys.path so utils/ is importable
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Subject      : {SUBJECT}")
print(f"Task         : {TASK}")
print(f"TR           : {TR} s")
print(f"HRF model    : {HRF_MODEL}")
print(f"High-pass    : {HIGH_PASS} s")
print(f"Output dir   : {OUTPUT_DIR}")

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})

try:
    import nibabel as nib
    print(f"nibabel  {nib.__version__}")
except ImportError:
    print("WARNING: nibabel not installed. Run: pip install nibabel")
    nib = None

try:
    import nilearn
    from nilearn.glm.first_level import make_first_level_design_matrix, FirstLevelModel
    from nilearn import plotting as nilearn_plotting
    print(f"nilearn  {nilearn.__version__}")
except ImportError:
    print("WARNING: nilearn not installed. Run: pip install nilearn")
    nilearn = None

try:
    from utils.io_utils import load_tsv, save_tsv, ensure_dir
    from utils.plotting import plot_design_matrix
    print("utils    loaded")
except ImportError:
    warnings.warn("utils not found — using pandas/matplotlib directly.")
    load_tsv = lambda p, **kw: pd.read_csv(p, sep="\t", **kw)
    save_tsv = lambda df, p: df.to_csv(p, sep="\t", index=False) or os.path.abspath(p)
    ensure_dir = lambda p: os.makedirs(p, exist_ok=True) or p
    plot_design_matrix = None

print("\nAll imports complete.")

## 3  Loading Preprocessed BOLD Data

We load the fMRIPrep-preprocessed BOLD NIfTI image with nibabel and inspect its shape (x, y, z, time) and the TR stored in the NIfTI header. If the real file is not present, we generate a small synthetic 4-D image to demonstrate the pipeline.

In [ ]:
USE_SYNTHETIC = False

if nib is not None and os.path.isfile(BOLD_PATH):
    bold_img = nib.load(BOLD_PATH)
    print(f"Loaded BOLD: {BOLD_PATH}")
else:
    USE_SYNTHETIC = True
    if not os.path.isfile(BOLD_PATH):
        print(f"BOLD file not found: {BOLD_PATH}")
    print("Generating synthetic 4-D BOLD image for demonstration...")

    rng = np.random.default_rng(seed=42)
    # Small brain volume: 20x20x10 voxels, 200 timepoints
    n_x, n_y, n_z, n_t = 20, 20, 10, 200
    bold_data = rng.standard_normal((n_x, n_y, n_z, n_t)).astype(np.float32) * 100 + 1000

    affine = np.diag([4.0, 4.0, 4.0, 1.0])   # 4 mm isotropic voxels
    header = nib.Nifti1Header()
    header.set_data_shape((n_x, n_y, n_z, n_t))
    header.set_zooms((4.0, 4.0, 4.0, TR))      # pixdim[4] = TR
    header.set_xyzt_units(xyz='mm', t='sec')
    bold_img = nib.Nifti1Image(bold_data, affine, header)
    print(f"  Synthetic BOLD shape : {bold_img.shape}")

bold_shape = bold_img.shape
n_scans = bold_shape[3]
header_tr = bold_img.header.get_zooms()[3]
if header_tr > 0:
    TR = float(header_tr)

print(f"\nBOLD shape   : {bold_shape}  (x, y, z, time)")
print(f"TR           : {TR} s")
print(f"Total time   : {n_scans * TR:.1f} s  ({n_scans} volumes)")
print(f"Voxel size   : {bold_img.header.get_zooms()[:3]} mm")

## 4  Loading and Inspecting the Events TSV

The events file (BIDS format) records the onset, duration, and trial type of every trial. For this emotion regulation task, trial types are:

| trial_type | Description |
|---|---|
| `Look_Neg` | Passively view negative images |
| `Look_Neu` | Passively view neutral images |
| `Reappraise` | Cognitively reappraise negative images |
| `Suppress` | Expressively suppress response to negative images |

Each row represents one trial. We will use these onsets and durations to construct the HRF-convolved design matrix regressors.

In [ ]:
if os.path.isfile(EVENTS_PATH):
    events_df = load_tsv(EVENTS_PATH)
    print(f"Loaded events: {EVENTS_PATH}")
else:
    print(f"Events file not found: {EVENTS_PATH}")
    print("Generating synthetic events for demonstration...")

    rng = np.random.default_rng(seed=42)
    total_duration = n_scans * TR  # seconds

    trial_types = ["Look_Neg", "Look_Neu", "Reappraise", "Suppress"]
    # 8 trials per condition, 8 s duration, spaced by 14 s ISI
    onsets, durations, ttypes = [], [], []
    current_onset = 10.0  # initial fixation

    for i in range(32):
        onsets.append(round(current_onset, 2))
        durations.append(8.0)
        ttypes.append(trial_types[i % 4])
        current_onset += 8.0 + rng.integers(10, 18)  # jittered ISI 10-18 s
        if current_onset >= total_duration - 10:
            break

    events_df = pd.DataFrame({
        "onset":      onsets,
        "duration":   durations,
        "trial_type": ttypes,
    })
    print(f"  Generated {len(events_df)} trials")

print(f"\nEvents shape : {events_df.shape}")
print(f"Columns      : {list(events_df.columns)}")
print(f"\nTrial type counts:")
print(events_df["trial_type"].value_counts().to_string())
print(f"\nFirst 8 rows:")
events_df.head(8)

In [ ]:
# Plot trial onsets and durations as a timeline
fig, ax = plt.subplots(figsize=(14, 3))

palette = {
    "Look_Neg":   "#e41a1c",
    "Look_Neu":   "#377eb8",
    "Reappraise": "#4daf4a",
    "Suppress":   "#ff7f00",
}
y_positions = {"Look_Neg": 3, "Look_Neu": 2, "Reappraise": 1, "Suppress": 0}

for _, row in events_df.iterrows():
    tt = row["trial_type"]
    ax.barh(
        y=y_positions.get(tt, 0),
        width=row["duration"],
        left=row["onset"],
        height=0.6,
        color=palette.get(tt, "grey"),
        alpha=0.85,
        edgecolor="white",
        linewidth=0.5,
    )

ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels(list(y_positions.keys()))
ax.set_xlabel("Time (s)")
ax.set_title("Trial Timeline — Emotion Regulation Task")

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=t) for t, c in palette.items()]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)
fig.tight_layout()
plt.show()

## 5  Building the Design Matrix

`make_first_level_design_matrix()` takes the frame times (one entry per TR), the events DataFrame, and configuration parameters, and returns a DataFrame where each row is a timepoint and each column is a regressor.

Key parameters:
- `frame_times` — array of timepoint centres in seconds (0, TR, 2·TR, …)
- `events` — events DataFrame with onset, duration, trial_type columns
- `hrf_model` — HRF to convolve with; `'spm'` is the canonical SPM HRF
- `drift_model` — `'cosine'` adds discrete cosine transform regressors
- `high_pass` — high-pass cutoff period in seconds (regressors below this frequency are included)
- `add_regs` — additional confound regressors DataFrame

In [ ]:
if nilearn is None:
    print("nilearn is required for this section. Run: pip install nilearn")
else:
    # Frame times: centre of each TR acquisition
    frame_times = np.arange(n_scans) * TR

    # Build design matrix without confounds first (confounds added in Section 8)
    design_matrix = make_first_level_design_matrix(
        frame_times=frame_times,
        events=events_df,
        hrf_model=HRF_MODEL,
        drift_model="cosine",
        high_pass=1.0 / HIGH_PASS,
        drift_order=1,
    )

    print(f"Design matrix shape : {design_matrix.shape}")
    print(f"  → {design_matrix.shape[0]} timepoints × {design_matrix.shape[1]} regressors")
    print(f"\nRegressor names:")
    for i, col in enumerate(design_matrix.columns):
        print(f"  [{i:2d}] {col}")

## 6  Visualising the Design Matrix

A heatmap of the design matrix shows each regressor as a column. The HRF-convolved condition regressors display the characteristic rise-and-fall pattern at trial times. Drift and confound regressors appear as slowly-varying sinusoidal patterns at the right of the matrix.

We use `utils.plotting.plot_design_matrix()` which wraps nilearn's `plot_design_matrix()` in a consistent figure style.

In [ ]:
if nilearn is not None:
    if plot_design_matrix is not None:
        fig = plot_design_matrix(design_matrix)
    else:
        # Fallback: call nilearn directly
        from nilearn.plotting import plot_design_matrix as _pdm
        fig, ax = plt.subplots(figsize=(10, 6))
        _pdm(design_matrix, ax=ax)
        ax.set_title("GLM Design Matrix (without confounds)")
        fig.tight_layout()
        plt.show()

## 7  Selecting Confound Regressors

fMRIPrep outputs a `*_desc-confounds_timeseries.tsv` file with many potential nuisance regressors. We use the same three-tier strategy from Module 09:

| Strategy | Columns |
|---|---|
| **minimal** | 6 motion params + FD |
| **moderate** | minimal + global signal, WM, CSF + 6 aCompCor |
| **aggressive** | moderate + 24-param motion + all aCompCor |

We use the **moderate** strategy as default. The first row of the confounds TSV is typically `NaN` for derivative-based columns (e.g., FD is undefined for volume 0); we fill these with 0.

In [ ]:
STRATEGY = "moderate"  # change to 'minimal' or 'aggressive' as needed

MOTION_6 = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
TISSUE    = ["global_signal", "white_matter", "csf"]
ACOMPCOR6 = [f"a_comp_cor_{i:02d}" for i in range(6)]

STRATEGY_COLS = {
    "minimal":    MOTION_6 + ["framewise_displacement"],
    "moderate":   MOTION_6 + ["framewise_displacement"] + TISSUE + ACOMPCOR6,
    "aggressive": (
        MOTION_6
        + [f"{p}_derivative1"        for p in MOTION_6]
        + [f"{p}_power2"             for p in MOTION_6]
        + [f"{p}_derivative1_power2" for p in MOTION_6]
        + ["framewise_displacement"] + TISSUE + ACOMPCOR6
    ),
}

if os.path.isfile(CONFOUNDS_PATH):
    confounds_raw = load_tsv(CONFOUNDS_PATH)
    print(f"Loaded confounds: {CONFOUNDS_PATH}")
    print(f"  {confounds_raw.shape[0]} volumes × {confounds_raw.shape[1]} columns available")
else:
    print(f"Confounds file not found: {CONFOUNDS_PATH}")
    print("Generating synthetic confounds...")
    rng = np.random.default_rng(seed=42)
    conf_dict = {col: rng.standard_normal(n_scans) * 0.1 for col in STRATEGY_COLS["moderate"]}
    # FD = 0 for first volume (undefined for derivative)
    conf_dict["framewise_displacement"][0] = np.nan
    for i in range(6):
        conf_dict[ACOMPCOR6[i]] = rng.standard_normal(n_scans)
    confounds_raw = pd.DataFrame(conf_dict)
    print(f"  Generated synthetic confounds: {confounds_raw.shape}")

desired_cols = STRATEGY_COLS[STRATEGY]
available_cols = [c for c in desired_cols if c in confounds_raw.columns]
missing_cols   = [c for c in desired_cols if c not in confounds_raw.columns]

if missing_cols:
    warnings.warn(f"  Missing columns (skipped): {missing_cols}")

confounds_clean = confounds_raw[available_cols].copy()

# Fill NaN values (first-volume derivatives are undefined → set to 0)
n_nan_before = confounds_clean.isna().sum().sum()
confounds_clean = confounds_clean.fillna(0)

print(f"\nStrategy         : {STRATEGY}")
print(f"Columns selected : {len(available_cols)}")
print(f"NaN values filled: {n_nan_before}")
print(f"Confound matrix  : {confounds_clean.shape}")
confounds_clean.head()

In [ ]:
# Check for high-motion volumes
if "framewise_displacement" in confounds_raw.columns:
    fd = pd.to_numeric(confounds_raw["framewise_displacement"], errors="coerce")
    n_flagged = int((fd > FD_THRESHOLD).sum())
    pct_flagged = 100.0 * n_flagged / n_scans
    print(f"FD > {FD_THRESHOLD} mm : {n_flagged} volumes ({pct_flagged:.1f}%)")
    if pct_flagged > 20:
        print("  ⚠  WARNING: >20% high-motion volumes — consider excluding this run.")

    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.plot(fd.values, color="steelblue", lw=1, label="FD")
    ax.axhline(FD_THRESHOLD, color="red", lw=1, ls="--", label=f"Threshold ({FD_THRESHOLD} mm)")
    ax.fill_between(range(n_scans), fd.values, FD_THRESHOLD,
                    where=(fd.values > FD_THRESHOLD), alpha=0.3, color="red")
    ax.set_xlabel("Volume")
    ax.set_ylabel("FD (mm)")
    ax.set_title(f"Framewise Displacement — {n_flagged} volumes above threshold")
    ax.legend()
    fig.tight_layout()
    plt.show()

## 8  Fitting the First-Level GLM

`FirstLevelModel` from nilearn fits the GLM at every voxel simultaneously using vectorised operations. We pass it the preprocessed BOLD image, the events DataFrame, and the confounds matrix. Under the hood, nilearn:

1. Constructs the design matrix (HRF convolution + drift + confounds)
2. Prewhitens the BOLD data and design matrix with an AR(1) model
3. Computes the OLS parameter estimates $\hat{\boldsymbol{\beta}}$ at each voxel
4. Estimates residual variance $\hat{\sigma}^2$ for contrast inference

The fitted model stores the full design matrix and $\hat{\boldsymbol{\beta}}$ maps for every regressor.

In [ ]:
if nilearn is None:
    print("nilearn is required for this section. Run: pip install nilearn")
    glm = None
else:
    print("Fitting FirstLevelModel...")
    glm = FirstLevelModel(
        t_r=TR,
        hrf_model=HRF_MODEL,
        drift_model="cosine",
        high_pass=1.0 / HIGH_PASS,
        standardize=False,
        signal_scaling=False,
        noise_model="ar1",
        n_jobs=1,
        verbose=0,
    )

    try:
        glm.fit(
            run_imgs=bold_img,
            events=events_df,
            confounds=confounds_clean,
        )
        print("GLM fit complete.")
        fitted_dm = glm.design_matrices_[0]
        print(f"Final design matrix: {fitted_dm.shape}")
        print(f"Condition regressors: {[c for c in fitted_dm.columns if c in events_df['trial_type'].unique()]}")
    except Exception as exc:
        print(f"ERROR fitting GLM: {exc}")
        glm = None

In [ ]:
# Visualise the fitted design matrix (includes confounds)
if glm is not None:
    fitted_dm = glm.design_matrices_[0]
    if plot_design_matrix is not None:
        fig = plot_design_matrix(fitted_dm)
    else:
        from nilearn.plotting import plot_design_matrix as _pdm
        fig, ax = plt.subplots(figsize=(12, 6))
        _pdm(fitted_dm, ax=ax)
        ax.set_title("Full Design Matrix (conditions + confounds + drift)")
        fig.tight_layout()
        plt.show()

## 9  Specifying Contrasts

Contrasts compare the parameter estimates ($\hat{\boldsymbol{\beta}}$) of different conditions. We specify contrasts as either:

1. **String expressions** — nilearn parses these into weight vectors automatically (e.g., `"Reappraise - Look_Neg"`)
2. **numpy arrays** — explicit weight vectors over all design matrix columns

For the emotion regulation task, our primary contrasts are:

| Contrast | Interpretation |
|---|---|
| `Reappraise - Look_Neg` | Regions engaged more during reappraisal than passive viewing of negatives |
| `Suppress - Look_Neg` | Regions engaged more during suppression than passive viewing of negatives |
| `Reappraise - Suppress` | Regions showing differential engagement between the two regulation strategies |

In [ ]:
contrasts = {
    "Reappraise_vs_Look_Neg":  "Reappraise - Look_Neg",
    "Suppress_vs_Look_Neg":    "Suppress - Look_Neg",
    "Reappraise_vs_Suppress":  "Reappraise - Suppress",
}

print("Contrasts defined:")
for name, expr in contrasts.items():
    print(f"  {name:30s}: {expr}")

# Check that contrast condition names exist in the fitted design matrix
if glm is not None:
    dm_cols = set(glm.design_matrices_[0].columns)
    required_conds = {"Look_Neg", "Reappraise", "Suppress"}
    found = required_conds & dm_cols
    missing = required_conds - dm_cols
    print(f"\nConditions in design matrix : {found}")
    if missing:
        print(f"  WARNING: missing conditions: {missing}")

## 10  Computing Contrast Maps

`glm.compute_contrast()` evaluates the contrast at every voxel and returns a NIfTI image of the statistic of your choice:

- `stat_type='z_score'` — z-statistic map (commonly used for display and group analysis)
- `stat_type='t'` — raw t-statistic map
- `output_type='effect_size'` — $\hat{c} = \mathbf{c}'\hat{\boldsymbol{\beta}}$

We compute z-maps for all three contrasts.

In [ ]:
z_maps = {}

if glm is None:
    print("GLM not fitted — skipping contrast computation.")
else:
    for contrast_name, contrast_expr in contrasts.items():
        try:
            z_map = glm.compute_contrast(
                contrast_def=contrast_expr,
                stat_type="t",
                output_type="z_score",
            )
            z_maps[contrast_name] = z_map
            z_data = z_map.get_fdata()
            finite_vals = z_data[np.isfinite(z_data)]
            print(f"  {contrast_name:30s} : shape {z_map.shape}, "
                  f"z range [{finite_vals.min():.2f}, {finite_vals.max():.2f}]")
        except Exception as exc:
            print(f"  ERROR computing {contrast_name}: {exc}")

    print(f"\nComputed {len(z_maps)} contrast map(s).")

## 11  Visualising Contrast Results

We use `nilearn.plotting.plot_stat_map()` to display each z-map on a glass brain and as axial slices. A typical threshold for display is |z| > 2.3 (equivalent to uncorrected *p* < 0.01, one-tailed), but real analyses should use family-wise error (FWE) or false discovery rate (FDR) correction.

> **Note:** The z-maps from synthetic data contain noise rather than meaningful activations. In real data you would expect prefrontal and parietal activity for Reappraise > Look_Neg.

In [ ]:
if nilearn is not None and z_maps:
    threshold = 2.3  # uncorrected z > 2.3 ≈ p < 0.01

    for contrast_name, z_map in z_maps.items():
        try:
            display = nilearn_plotting.plot_stat_map(
                stat_map_img=z_map,
                threshold=threshold,
                display_mode="z",
                cut_coords=5,
                title=f"{contrast_name}  (|z| > {threshold})",
                colorbar=True,
                cmap="cold_hot",
            )
            nilearn_plotting.show()
        except Exception as exc:
            print(f"  Could not display {contrast_name}: {exc}")
else:
    print("No z-maps available to visualise.")

## 12  Saving Contrast Maps

We save each z-map as a gzip-compressed NIfTI file (`.nii.gz`) to the output directory. These files are then used as inputs to the group-level analysis in Module 11.

Output naming convention: `{subject}_task-{task}_contrast-{name}_stat-z_statmap.nii.gz`

In [ ]:
if nib is None:
    print("nibabel is required to save NIfTI files. Run: pip install nibabel")
elif not z_maps:
    print("No z-maps to save.")
else:
    subject_out_dir = os.path.join(OUTPUT_DIR, SUBJECT, "func")
    ensure_dir(subject_out_dir)
    print(f"Saving z-maps to: {subject_out_dir}")

    saved_files = []
    for contrast_name, z_map in z_maps.items():
        out_fname = (
            f"{SUBJECT}_task-{TASK}_contrast-{contrast_name}_stat-z_statmap.nii.gz"
        )
        out_path = os.path.join(subject_out_dir, out_fname)
        try:
            nib.save(z_map, out_path)
            saved_files.append(out_path)
            print(f"  Saved: {out_fname}")
        except Exception as exc:
            print(f"  ERROR saving {contrast_name}: {exc}")

    print(f"\n{len(saved_files)} file(s) saved successfully.")

## 13  Quality Checks

Before trusting the GLM results, it is good practice to inspect several properties of the design matrix:

1. **Rank** — the design matrix must be full column rank (rank = number of columns). A rank-deficient matrix means the model is not identifiable (collinear regressors).
2. **Condition coverage** — each condition regressor should have sufficient variance. A regressor with near-zero variance (e.g., a condition with very few trials) inflates the standard error of its contrast.
3. **Variance Inflation Factor (VIF)** — measures multicollinearity. VIF > 5–10 indicates that a regressor is nearly a linear combination of others, making the estimate unreliable.

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the coefficient of determination of regressing column $j$ onto all other columns.

In [ ]:
if glm is not None:
    dm_check = glm.design_matrices_[0].copy()
    dm_array = dm_check.values

    # -----------------------------------------------------------------------
    # 1. Rank check
    # -----------------------------------------------------------------------
    rank = np.linalg.matrix_rank(dm_array)
    n_cols = dm_array.shape[1]
    rank_ok = rank == n_cols
    print("=" * 55)
    print(f"  Design Matrix Quality Check")
    print("=" * 55)
    print(f"  Shape         : {dm_array.shape}")
    print(f"  Rank          : {rank} / {n_cols}  {'✓ Full rank' if rank_ok else '✗ RANK DEFICIENT — check for collinear regressors!'}")

    # -----------------------------------------------------------------------
    # 2. Condition coverage (standard deviation of condition regressors)
    # -----------------------------------------------------------------------
    cond_cols = [c for c in dm_check.columns if c in events_df["trial_type"].unique()]
    print(f"\n  Condition regressor standard deviations:")
    for col in cond_cols:
        std_val = dm_check[col].std()
        flag = "  ⚠ LOW VARIANCE" if std_val < 0.01 else ""
        print(f"    {col:20s}: std = {std_val:.4f}{flag}")

    # -----------------------------------------------------------------------
    # 3. VIF for condition regressors
    # -----------------------------------------------------------------------
    from numpy.linalg import lstsq

    print(f"\n  VIF for condition regressors (threshold = 5):")
    for col in cond_cols:
        y_vif = dm_check[col].values
        X_vif = dm_check.drop(columns=[col]).values
        # Add intercept column
        X_vif = np.column_stack([X_vif, np.ones(len(y_vif))])
        try:
            coefs, _, _, _ = lstsq(X_vif, y_vif, rcond=None)
            y_pred = X_vif @ coefs
            ss_res = np.sum((y_vif - y_pred) ** 2)
            ss_tot = np.sum((y_vif - y_vif.mean()) ** 2)
            r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
            r2 = min(r2, 1.0 - 1e-10)  # avoid division by zero
            vif = 1.0 / (1.0 - r2)
            flag = "  ⚠ HIGH VIF" if vif > 5 else ""
            print(f"    {col:20s}: VIF = {vif:6.2f}{flag}")
        except Exception:
            print(f"    {col:20s}: VIF = could not compute")

    print("=" * 55)
else:
    print("GLM not fitted — skipping quality checks.")

In [ ]:
# Correlation heatmap of condition regressors
if glm is not None and cond_cols:
    corr = dm_check[cond_cols].corr()

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(cond_cols)))
    ax.set_yticks(range(len(cond_cols)))
    ax.set_xticklabels(cond_cols, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(cond_cols, fontsize=9)
    plt.colorbar(im, ax=ax, label="Pearson r")
    ax.set_title("Condition Regressor Correlations")

    for i in range(len(cond_cols)):
        for j in range(len(cond_cols)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if abs(corr.values[i, j]) > 0.5 else "black")

    fig.tight_layout()
    plt.show()

## 14  Summary

In this module we covered the complete first-level GLM pipeline:

| Step | Tool / Method | Output |
|---|---|---|
| Load BOLD | `nibabel.load()` | 4-D NIfTI image |
| Load events | `utils.io_utils.load_tsv()` | Events DataFrame |
| Build design matrix | `make_first_level_design_matrix()` | Design matrix DataFrame |
| Select confounds | Strategy-based column selection | Confounds DataFrame |
| Fit GLM | `FirstLevelModel.fit()` | Fitted model object |
| Define contrasts | String expressions | Contrast definitions |
| Compute z-maps | `glm.compute_contrast()` | NIfTI z-maps |
| Save outputs | `nibabel.save()` | `.nii.gz` files |
| Quality check | Rank, VIF, correlation | Diagnostic plots |

### Key Takeaways

- The design matrix encodes experiment timing, HRF shape, and nuisance regressors in a single matrix
- HRF convolution models the delayed vascular response to neural activity
- Confound regression removes structured noise; choose strategy based on data quality and research question
- Always quality-check the design matrix rank and regressor correlations before computing contrasts
- Z-maps from first-level analyses feed into group-level random-effects models (Module 11)

### Next Steps

The z-maps saved in this module are the input to **Module 11: Group-Level Analysis**, where we combine first-level contrast maps across subjects to make population-level inferences. The scripts `prepare_glm_regressors.py` and `run_first_level_glm.py` in `scripts/` automate the full pipeline for batch processing across multiple subjects.